<a href="https://colab.research.google.com/github/aaditjoshi-star/Aadit/blob/main/Exercise1_Prompt_Chaining_Customer_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI
**Aadit Joshi** | Agentic AI and Distributed Systems | San José State University | Fall 2026

**Goal:** Build a prompt chain that simulates a customer service flow. The output of each prompt becomes the input of the next one.

**Tools used**
- Google Colab (Python 3)
- Gemini API through the `google-genai` Python SDK, model `gemini-3.8-flash` (free tier)
- Plain Python for JSON parsing, policy lookup, the escalation rules and output checks; `pandas` for the results table

**How to run**
1. Create a free API key at https://aistudio.google.com/apikey
2. In Colab, open **Secrets** (key icon in the left sidebar), add `GEMINI_API_KEY`, and turn on notebook access.
3. Runtime > Run all.

## Chain design

```
Customer message
   |
   v
STEP 1  Classify + extract ........ JSON case record: category, urgency, sentiment, order_id,
   |                                amount, missing_info ...
   |    (category picks the policy excerpts for Step 3; missing_info drives Step 2)
   v
STEP 2  Gather missing info ....... asks ONLY for the details listed in missing_info
   |    (the customer's answers are added to the case)
   v
STEP 3  Propose solution .......... JSON plan grounded in the retrieved policies, with a confidence score
   |
   v
STEP 4  Escalation rule + reply ... a rules engine reads Step 1 + Step 3 and decides RESOLVE or ESCALATE,
                                    then the final customer reply is written and checked against guardrails
```

The company, policies and tickets are fictional test data ("NimbusCart", an online electronics store).

In [26]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q google-genai

In [ ]:
# Setup: Gemini client and shared helper functions
import os, json, re, time, textwrap
from google import genai
from google.genai import types, errors

# API key is stored in Colab Secrets (key icon in the left sidebar) as GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass  # not running in Colab: set the GEMINI_API_KEY environment variable instead

client = genai.Client()  # reads GEMINI_API_KEY

# First-choice model plus free-tier fallbacks (used only if a model is unavailable or out of daily quota)
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite"]
model_index = 0
MODELS_USED = set()
RETRYABLE = {429, 500, 503}


def generate(contents, system=None, json_mode=False, max_retries=4):
    """Send a prompt (or a multi-turn history) to Gemini and return the full response.
    Retries on rate limits / server errors and falls back to the next model if needed."""
    global model_index
    config = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json" if json_mode else None,
    )
    attempt = 0
    while True:
        model = MODELS[model_index]
        try:
            response = client.models.generate_content(model=model, contents=contents, config=config)
            if response.text and response.text.strip():
                MODELS_USED.add(model)
                return response
            code, detail = "empty", "empty response"
        except errors.APIError as e:
            code, detail = e.code, str(e)
            if code not in RETRYABLE and code != 404:
                raise
        attempt += 1
        out_of_daily_quota = code == 429 and "perday" in detail.lower().replace(" ", "")
        if code == 404 or out_of_daily_quota or attempt > max_retries:
            if model_index + 1 >= len(MODELS):
                raise RuntimeError(f"All models failed. Last error: {detail[:300]}")
            model_index += 1
            attempt = 0
            print(f"   [{model} unavailable ({code}); switching to {MODELS[model_index]}]")
        else:
            wait = 15 * attempt
            print(f"   [{model} returned {code}; retrying in {wait}s]")
            time.sleep(wait)


def call_llm(prompt, system=None, json_mode=False):
    """Single-turn call that returns just the text."""
    return generate(prompt, system=system, json_mode=json_mode).text.strip()


def fill(template, **values):
    """Insert values into {placeholders} without tripping over the literal JSON braces in prompts."""
    for key, value in values.items():
        text = value if isinstance(value, str) else json.dumps(value, indent=2)
        template = template.replace("{" + key + "}", text)
    return template


def parse_json(text):
    """Parse a JSON reply, tolerating ```json fences or stray text around the object."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def wrap(text, width=100):
    """Wrap long lines so outputs are readable on GitHub (bullets keep a hanging indent)."""
    out = []
    for line in str(text).splitlines():
        if len(line) <= width:
            out.append(line)
        else:
            is_item = re.match(r"\s*(- |\* |\d+\. )", line)
            out.append(textwrap.fill(line, width, subsequent_indent="    " if is_item else ""))
    return "\n".join(out)


def show(title, body=""):
    print("\n" + "=" * 100)
    print(title)
    print("-" * 100)
    if body != "":
        print(wrap(body) if isinstance(body, str) else json.dumps(body, indent=2))

In [ ]:
# Quick connectivity check
print(call_llm("Reply with exactly one word: ready"))
print("Model in use:", MODELS[model_index])

## Test data: support policies and customer tickets

`POLICIES` is a small knowledge base. Step 3 only sees the excerpts for the category that Step 1 assigned (a simple retrieval step).

Each ticket has a scripted `followup`: the customer's answer to Step 2's questions. In a live system this would be the customer's real reply; scripting it keeps the notebook runnable end to end with **Run all**.

In [ ]:
POLICIES = {
    "shipping": [
        "SHIP-1: Standard orders arrive 3-7 business days after they ship. Tracking can pause for up to 3 business days while a package is at a carrier hub.",
        "SHIP-2: If tracking has not updated for 4 or more business days, support opens a carrier trace (answer within 2 business days). If the carrier cannot locate the package, the customer chooses a free replacement or a full refund.",
    ],
    "billing": [
        "BILL-1: A second charge that is still PENDING is usually an authorization hold and drops off within 3-5 business days.",
        "BILL-2: If a duplicate charge has POSTED, support refunds the duplicate. Refunds reach the customer's bank in 5-10 business days.",
        "BILL-3: Billing disputes over $500, or any case that mentions a chargeback, must be handled by the Billing Escalations team.",
    ],
    "returns_refunds": [
        "RET-1: Items can be returned within 30 days of delivery in original condition. Return shipping is free for defective items.",
        "RET-2: Refunds go to the original payment method within 5 business days after the return is received.",
    ],
    "technical": [
        "TECH-1: The SP-200 smart plug works only on 2.4 GHz Wi-Fi. It cannot join a 5 GHz-only network; most routers can turn on a 2.4 GHz band in their settings.",
        "TECH-2: After a firmware update, reset the SP-200 by holding its button for 10 seconds until the light blinks amber, then add it again in the NimbusHome app.",
        "TECH-3: If the device still fails after TECH-1 and TECH-2, it is replaced free under the 1-year warranty.",
    ],
    "account": [
        "ACCT-1: Customers reset passwords at nimbuscart.example/reset. Support never asks for passwords.",
    ],
    "other": [],
}
ALWAYS_INCLUDED = [
    "ESC-1: Escalated cases are picked up by a senior specialist within 4 business hours. The customer will not need to repeat their details.",
]


def retrieve_policies(category):
    """Retrieval step: return the policy excerpts for the category chosen in Step 1."""
    return POLICIES.get(category, []) + ALWAYS_INCLUDED


TICKETS = [
    {
        "id": "T1",
        "message": "Hi, I ordered a laptop stand about two weeks ago and it still hasn't arrived. "
                   "The tracking page hasn't changed in 5 days. Can you check what's going on?",
        "followup": "Sure. The order number is NC-48213 and I placed it on September 8.",
    },
    {
        "id": "T2",
        "message": "This is ridiculous. You charged me TWICE for order NC-50991, $649.99 each time! "
                   "I want my money back today or I'm filing a chargeback with my bank.",
        "followup": "Both charges POSTED on 9/20, they are not pending. Visa ending in 4417.",
    },
    {
        "id": "T3",
        "message": "My smart plug stopped connecting to wifi after last night's update. "
                   "I've unplugged it a few times already. Pretty annoying.",
        "followup": "It's the SP-200. My router is set to 5 GHz only I think. The app just says 'Device offline'.",
    },
]
print(f"{len(TICKETS)} test tickets loaded; {sum(len(v) for v in POLICIES.values())} policy excerpts.")

## Step 1: Classify and extract

### Iteration 1 (v1): a first, minimal prompt
My first attempt at Step 1 was a one-line instruction. The test below runs it on all three tickets and checks whether the output can be passed to Step 2 as structured data.

In [ ]:
STEP1_PROMPT_V1 = "Classify this customer support message:\n\n{message}"

v1_results = {}
for t in TICKETS:
    raw = call_llm(fill(STEP1_PROMPT_V1, message=t["message"]))
    try:
        parse_json(raw)
        parseable = True
    except Exception:
        parseable = False
    v1_results[t["id"]] = {"parseable_json": parseable, "words": len(raw.split())}
    show(f"{t['id']} | Step 1 v1 output (parseable as JSON: {parseable}, {len(raw.split())} words)", raw[:900] + (" ..." if len(raw) > 900 else ""))

**What went wrong with v1 and what I changed for v2**

| Problem observed in v1 | Fix in v2 |
|---|---|
| Free-form prose/markdown, so the next step cannot read it reliably | JSON-only output with a fixed schema (plus Gemini JSON mode) |
| Category names vary between runs ("Shipping issue", "Delivery delay"...) | Closed list of allowed values for category, urgency and sentiment |
| No notion of what information is still missing | `missing_info` field, driven by a "required details per category" table |
| Model may guess details such as an order number | Explicit rule: use only facts in the message, `null` if not stated |
| No signals for escalation | `amount_usd` and `chargeback_or_legal_threat` fields that Step 4's rules read |

### Iteration 2 (v2): the prompt used in the final chain

In [ ]:
STEP1_SYSTEM = "You are the intake classifier for NimbusCart customer support. You output valid JSON only."

STEP1_PROMPT = """Classify the customer message and extract the key details.
Use ONLY information stated in the message. Use null when something is not stated. Never guess an order number.

Return JSON with exactly these keys:
{
  "category": one of ["shipping", "billing", "returns_refunds", "technical", "account", "other"],
  "urgency": one of ["low", "medium", "high"],
  "sentiment": one of ["positive", "neutral", "frustrated", "angry"],
  "summary": one sentence, max 25 words, third person,
  "order_id": string or null,
  "product": string or null,
  "amount_usd": number or null,
  "chargeback_or_legal_threat": true or false,
  "missing_info": list of details support still needs (max 3; empty list if none)
}

Required details by category (use these to fill missing_info):
- shipping: order_id, order date
- billing: order_id, charge date(s), whether the charges are pending or posted
- returns_refunds: order_id, product, reason for return
- technical: product model, router/network details, exact error message
- account: account email

Urgency guide: high = money lost, safety issue, or threat to dispute/leave; medium = order late or product not working; low = general question.

Customer message:
<message>
{message}
</message>"""

CASE_DEFAULTS = {"category": "other", "urgency": "medium", "sentiment": "neutral", "summary": "",
                 "order_id": None, "product": None, "amount_usd": None,
                 "chargeback_or_legal_threat": False, "missing_info": []}
ALLOWED = {"category": ["shipping", "billing", "returns_refunds", "technical", "account", "other"],
           "urgency": ["low", "medium", "high"],
           "sentiment": ["positive", "neutral", "frustrated", "angry"]}


def classify(message):
    """STEP 1: raw customer message -> validated JSON case record."""
    raw = parse_json(call_llm(fill(STEP1_PROMPT, message=message), system=STEP1_SYSTEM, json_mode=True))
    case = {**CASE_DEFAULTS, **{k: v for k, v in raw.items() if k in CASE_DEFAULTS}}
    for field, allowed in ALLOWED.items():          # guard against values outside the schema
        if case[field] not in allowed:
            case[field] = CASE_DEFAULTS[field]
    try:
        case["amount_usd"] = float(case["amount_usd"]) if case["amount_usd"] is not None else None
    except (TypeError, ValueError):
        case["amount_usd"] = None
    case["missing_info"] = list(case["missing_info"] or [])[:3]
    return case

## Step 2: Gather missing information
**Input from the previous step:** the Step 1 case record, especially `missing_info` and `sentiment`.
If `missing_info` is empty, the step is skipped in code (no API call).

In [ ]:
STEP2_SYSTEM = "You are a friendly, professional NimbusCart support agent."

STEP2_PROMPT = """Write a short message to the customer that asks ONLY for the details listed in "missing_info".

Case record from Step 1:
{case}

Constraints:
- Start with one sentence acknowledging the issue that fits the customer's sentiment ({sentiment}).
- Then ask one numbered question per missing detail (max 3).
- Do not propose a solution yet and do not promise refunds, dates or outcomes.
- Never ask for a full card number or a password.
- Max 70 words. Plain text, no subject line, no sign-off."""


def ask_for_missing_info(case):
    """STEP 2: case record -> clarifying questions (or None when nothing is missing)."""
    if not case["missing_info"]:
        return None
    return call_llm(fill(STEP2_PROMPT, case=case, sentiment=case["sentiment"]), system=STEP2_SYSTEM)

## Step 3: Propose a solution
**Input from previous steps:** the Step 1 case record, the Step 2 questions, the customer's answers, and the policy excerpts retrieved using Step 1's `category`.

In [ ]:
STEP3_SYSTEM = "You are a senior NimbusCart support specialist. You output valid JSON only."

STEP3_PROMPT = """Propose a resolution for this case using ONLY the policy excerpts below.
If the policies do not cover the case, say so and set needs_human to true.

Case record (Step 1):
{case}

Questions we asked the customer (Step 2):
{questions}

Customer's answers:
<answers>
{followup}
</answers>

Policy excerpts retrieved for category "{category}":
{policies}

Return JSON with exactly these keys:
{
  "diagnosis": "1-2 sentences on what most likely happened",
  "resolution_steps": ["2 to 4 concrete actions, each saying who does what"],
  "policy_cited": ["policy IDs you relied on, e.g. SHIP-2"],
  "customer_commitments": ["only what a cited policy allows us to promise"],
  "confidence": number between 0 and 1 that this resolves the issue without a human,
  "needs_human": true or false,
  "needs_human_reason": string or null
}"""


def propose_solution(case, questions, followup):
    """STEP 3: case + Q&A + retrieved policies -> JSON resolution plan."""
    policies = retrieve_policies(case["category"])
    prompt = fill(STEP3_PROMPT, case=case, questions=questions or "(none needed)",
                  followup=followup if questions else "(no follow-up needed)",
                  category=case["category"], policies="\n".join(policies))
    return parse_json(call_llm(prompt, system=STEP3_SYSTEM, json_mode=True))

## Step 4: Escalation rule and final reply
**Input from previous steps:** Step 1 (sentiment, urgency, amount, chargeback flag) and Step 3 (confidence, needs_human).

The escalation decision is made by **code**, not by the model, so it is consistent and auditable. The model then writes the reply for that decision. The reply is checked against guardrails; if a rule is broken, the model gets one retry with the specific problems listed.

In [ ]:
ESCALATION_RULES = [
    ("angry customer with high urgency", lambda c, s: c["sentiment"] == "angry" and c["urgency"] == "high"),
    ("amount over $500",                 lambda c, s: (c["amount_usd"] or 0) > 500),
    ("chargeback or legal threat",       lambda c, s: bool(c["chargeback_or_legal_threat"])),
    ("solution confidence below 0.7",    lambda c, s: float(s.get("confidence") or 0) < 0.7),
    ("Step 3 flagged needs_human",       lambda c, s: bool(s.get("needs_human"))),
]


def decide_escalation(case, solution):
    """Rules engine: returns ("ESCALATE", reasons) if any rule fires, else ("RESOLVE", [])."""
    reasons = [name for name, rule in ESCALATION_RULES if rule(case, solution)]
    return ("ESCALATE" if reasons else "RESOLVE"), reasons


STEP4_SYSTEM = "You are a NimbusCart support agent writing the final reply to a customer."

STEP4_PROMPT = """Write the final reply to the customer.

Case record (Step 1):
{case}

Customer's answers (Step 2):
<answers>
{followup}
</answers>

Proposed resolution (Step 3):
{solution}

Decision from the escalation rules: {decision}
{decision_instructions}

Rules for the reply:
- One sentence of empathy that fits the customer's sentiment ({sentiment}); warm, professional, no blame.
- Only commit to what is listed in customer_commitments. Never invent dates, amounts or policies.
- No exclamation marks. Plain text. 120 words maximum.
- End with the line: NimbusCart Support"""

DECISION_INSTRUCTIONS = {
    "RESOLVE": "Explain the resolution steps in plain language and tell the customer exactly what happens next.",
    "ESCALATE": ("Tell the customer a senior specialist will take over within 4 business hours (policy ESC-1), "
                 "briefly restate what we already know so they will not need to repeat it, and do not promise any outcome."),
}


def check_reply(text):
    """Guardrails for the final reply."""
    problems = []
    words = len(text.split())
    if words > 120:
        problems.append(f"{words} words (limit is 120)")
    if not text.rstrip().rstrip(".").endswith("NimbusCart Support"):
        problems.append('does not end with the line "NimbusCart Support"')
    if "!" in text:
        problems.append("contains an exclamation mark")
    return problems


def write_reply(case, followup, solution, decision, reasons, max_attempts=2):
    """STEP 4: decision + everything above -> final customer reply (checked, with one retry)."""
    prompt = fill(STEP4_PROMPT, case=case, followup=followup or "(none)", solution=solution,
                  decision=decision + (f" (reasons: {'; '.join(reasons)})" if reasons else ""),
                  decision_instructions=DECISION_INSTRUCTIONS[decision], sentiment=case["sentiment"])
    reply = call_llm(prompt, system=STEP4_SYSTEM)
    attempts = 1
    while check_reply(reply) and attempts < max_attempts:
        problems = check_reply(reply)
        print(f"   [guardrail check failed: {problems}; asking the model to fix it]")
        reply = call_llm(prompt + "\n\nYour previous draft broke these rules: " + "; ".join(problems)
                         + "\nPrevious draft:\n" + reply + "\n\nRewrite it so it follows every rule.",
                         system=STEP4_SYSTEM)
        attempts += 1
    return reply, attempts

## Run the full chain on all three tickets
Each block below shows what went **into** a step and what came **out**, so the hand-off between steps is visible.

In [ ]:
def run_chain(ticket):
    show(f"TICKET {ticket['id']} | customer message", ticket["message"])

    case = classify(ticket["message"])
    show("STEP 1 | classify + extract   (input: raw customer message)", case)

    questions = ask_for_missing_info(case)
    if questions:
        show(f"STEP 2 | clarifying questions   (input: Step 1 missing_info = {case['missing_info']})", questions)
        show("        simulated customer reply (scripted test data)", ticket["followup"])
        followup = ticket["followup"]
    else:
        show("STEP 2 | skipped: Step 1 found no missing information")
        followup = None

    solution = propose_solution(case, questions, followup)
    show(f"STEP 3 | proposed solution   (input: Step 1 case + Step 2 Q&A + {case['category']} policies)", solution)

    decision, reasons = decide_escalation(case, solution)
    show("STEP 4a | escalation rules   (input: Step 1 signals + Step 3 confidence/needs_human)",
         f"Decision: {decision}" + (f"\nRules that fired: {reasons}" if reasons else "\nNo rules fired."))

    reply, attempts = write_reply(case, followup, solution, decision, reasons)
    show(f"STEP 4b | final customer reply   (guardrail checks: {check_reply(reply) or 'all passed'})", reply)

    if decision == "ESCALATE":
        handoff = {"ticket": ticket["id"], "category": case["category"], "urgency": case["urgency"],
                   "order_id": case["order_id"], "amount_usd": case["amount_usd"], "summary": case["summary"],
                   "customer_details": followup, "suggested_steps": solution.get("resolution_steps"),
                   "escalation_reasons": reasons}
        show("STEP 4c | internal hand-off note for the senior specialist (built from Steps 1-3)", handoff)

    return {"ticket": ticket["id"], "category": case["category"], "urgency": case["urgency"],
            "sentiment": case["sentiment"], "missing_info": ", ".join(case["missing_info"]) or "(none)",
            "policies_cited": ", ".join(solution.get("policy_cited", [])),
            "confidence": solution.get("confidence"), "decision": decision,
            "reply_words": len(reply.split()), "reply_attempts": attempts,
            "guardrails_passed": not check_reply(reply)}


results = [run_chain(t) for t in TICKETS]

## Results summary and before/after comparison for Step 1

In [ ]:
import pandas as pd

summary = pd.DataFrame(results)
summary["step1_v1_parseable"] = summary["ticket"].map(lambda t: v1_results[t]["parseable_json"])
summary["step1_v2_parseable"] = True  # every v2 output above was parsed and validated, or the cell would have failed
display(summary)
print("Model(s) used:", ", ".join(sorted(MODELS_USED)))

## Notes

**How each step uses the previous output**
- **Step 1** turns an unstructured message into a JSON case record. Its `category` chooses which policies Step 3 sees, and its `missing_info` list decides what Step 2 asks.
- **Step 2** only asks for what Step 1 marked as missing, in a tone matched to Step 1's `sentiment`. The customer's answers are added to the case.
- **Step 3** combines the case record, the Q&A from Step 2 and the retrieved policies to produce a grounded plan with a confidence score and `needs_human` flag.
- **Step 4** applies deterministic escalation rules to Step 1's signals and Step 3's confidence, then writes the final reply for that decision. Escalated tickets also get an internal hand-off note so the specialist does not start from zero.

**Testing and iteration**
- Step 1 v1 (one-line prompt) returned free-form text; the `step1_v1_parseable` column above shows whether any of it could be handed to the next step as structured data. v2 added a JSON schema, closed value lists, a missing-info table and a "do not guess" rule.
- The three test tickets cover the three paths: resolve after asking for info (T1), escalate on amount + chargeback + anger (T2), and technical troubleshooting grounded in a policy (T3).
- Step 4 has automatic guardrail checks (length, sign-off, no exclamation marks) with one self-correcting retry.

**Limitations:** the customer follow-ups are scripted, the knowledge base is tiny, and category-based retrieval would be replaced by embedding search in a real system.